# D2 - Domain-Adaptive Pretraining for Tamil

Continues masked-language-model pretraining of MuRIL on in-domain YouTube comments,
then fine-tunes the adapted model on the HopeEDI Tamil task and compares it against
the non-DAPT baseline established in the sweep.

**Why the romanized subset.** The HopeEDI Tamil task data is 84% romanized / 16%
native script. The collected corpus is 43% / 57% the other way. Adapting on the full
corpus would spend most of the budget on a script the task barely contains, so the
primary run uses the romanized + mixed subset (5,152 comments). Set `VARIANT = "full"`
to run the mismatched version as an ablation.

**The comparison must be fair.** The DAPT model gets the same treatment the sweep gave
every other model: a 3-point learning-rate grid selected on dev, then the winner re-run
at three seeds. Fine-tuning it at the non-DAPT model's best learning rate would be the
exact mistake the sweep was built to correct.

    Stage 0   DAPT          continued MLM, perplexity measured before and after
    Stage 1   grid          lr in {1e-5, 2e-5, 3e-5}, seed 42, selected on DEV
    Stage 2   replication   best config at seeds 1234 and 2026

**Baseline to beat:** MuRIL Tamil, no DAPT, 0.6359 +/- 0.0008 test macro F1.
Tamil's seed spread was 0.0342, so a gain smaller than that is inside the noise.

**Runtime:** roughly 15 min DAPT + 5 fine-tuning runs at ~30 min = about 2h45m.
Fully resumable - every run appends to Drive and completed work is skipped on restart,
so a disconnect costs at most one run.

In [ ]:
# ============================================================
# 0. Environment
# ============================================================
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate evaluate

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
BASE_MODEL = "google/muril-base-cased"   # the model that actually learns on Tamil
LANG       = "tamil"
VARIANT    = "romanized"                 # "romanized" (matches the task) or "full"

# --- DAPT (stage 0) ---
DAPT_EPOCHS   = 10
DAPT_LR       = 5e-5
DAPT_BATCH    = 16
MLM_PROB      = 0.15
HELDOUT_FRAC  = 0.05                     # for before/after perplexity

# --- fine-tuning (stages 1 and 2), identical to the sweep ---
LR_GRID      = [1e-5, 2e-5, 3e-5]
EPOCHS       = 8
PATIENCE     = 5
GRID_SEED    = 42
EXTRA_SEEDS  = [1234, 2026]
MAX_LEN, BATCH_SIZE, WEIGHT_DECAY = 128, 16, 0.01

# non-DAPT reference, from the sweep
BASELINE_MEAN, BASELINE_STD = 0.6359, 0.0008
SEED_SPREAD = 0.0342                     # Tamil noise floor

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, gc, shutil, time, math

BASE_DIR     = Path("/content/drive/MyDrive/hope_speech")
DATA_DIR     = BASE_DIR / "processed"
METRICS_OUT  = BASE_DIR / "metrics"
MODELS_OUT   = BASE_DIR / "models"
METRICS_OUT.mkdir(parents=True, exist_ok=True)
MODELS_OUT.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = METRICS_OUT / "dapt_results.json"
DAPT_DIR     = MODELS_OUT / f"muril-dapt-{LANG}-{VARIANT}"
CORPUS_PATH  = DATA_DIR / "tamil_corpus_clean.csv"

def load_results():
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text())
    return {"runs": [], "dapt": None}

def save_results(res):
    RESULTS_PATH.write_text(json.dumps(res, indent=2))

def already_done(res, lr, seed):
    return any(r["variant"] == VARIANT and abs(r["lr"] - lr) < 1e-12 and r["seed"] == seed
               for r in res["runs"])

results = load_results()
print(f"Config OK - variant '{VARIANT}' - {len(results['runs'])} fine-tuning runs recorded")
print(f"DAPT model dir: {DAPT_DIR}")
print(f"DAPT already trained: {DAPT_DIR.exists()}")
print(f"Planned fine-tuning runs: {len(LR_GRID) + len(EXTRA_SEEDS)}")

In [ ]:
# ============================================================
# 2. Build the DAPT corpus
# ============================================================
import pandas as pd
import numpy as np

corpus = pd.read_csv(CORPUS_PATH)
corpus["text"] = corpus["text"].fillna("").astype(str)
corpus = corpus[corpus["text"].str.strip().str.len() > 10]

print("Full collected corpus by kind:")
print(corpus["kind"].value_counts().to_string())

if VARIANT == "romanized":
    corpus = corpus[corpus["kind"].isin(["romanized", "mixed"])]
    why = "romanized + mixed only - matches the task's 84% romanized distribution"
else:
    why = "full corpus - script distribution does NOT match the task"

corpus = corpus.sample(frac=1.0, random_state=42).reset_index(drop=True)
n_held = max(50, int(len(corpus) * HELDOUT_FRAC))
dapt_eval  = corpus.iloc[:n_held]
dapt_train = corpus.iloc[n_held:]

print(f"\nVariant '{VARIANT}': {why}")
print(f"  DAPT train   {len(dapt_train):>6} comments")
print(f"  DAPT heldout {len(dapt_eval):>6} comments (for perplexity)")

# how much text is this, really?
tok_est = int(dapt_train["text"].str.split().str.len().sum())
print(f"  ~{tok_est:,} whitespace tokens  ({tok_est / 1e6:.2f}M)")
print("\nNote for the writeup: this is small for DAPT. Published DAPT studies use")
print("tens of millions of tokens. A null result here is as likely to reflect corpus")
print("size as it is to reflect the method.")

In [ ]:
# ============================================================
# 3. Stage 0 - DAPT (continued masked-language-model pretraining)
# ============================================================
from transformers import (AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling, set_seed)
from datasets import Dataset

def mlm_datasets(tok):
    out = {}
    for name, df in (("train", dapt_train), ("eval", dapt_eval)):
        d = Dataset.from_pandas(df[["text"]], preserve_index=False)
        out[name] = d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                          batched=True, remove_columns=["text"])
    return out

def perplexity(model, tok, ds):
    set_seed(42)   # fix the masking pattern so before/after are comparable
    args = TrainingArguments(output_dir="/content/ppl_tmp", per_device_eval_batch_size=32,
                             fp16=True, report_to="none", disable_tqdm=True)
    coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=MLM_PROB)
    t = Trainer(model=model, args=args, data_collator=coll, eval_dataset=ds)
    return math.exp(t.evaluate()["eval_loss"])

if DAPT_DIR.exists():
    print(f"DAPT model already exists at {DAPT_DIR} - skipping stage 0.")
    if results.get("dapt"):
        d = results["dapt"]
        print(f"  perplexity {d['ppl_before']:.2f} -> {d['ppl_after']:.2f}")
else:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    ds  = mlm_datasets(tok)

    print("Measuring perplexity BEFORE adaptation ...")
    model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
    ppl_before = perplexity(model, tok, ds["eval"])
    print(f"  base MuRIL perplexity on held-out in-domain text: {ppl_before:.2f}")

    print(f"\nRunning DAPT: {DAPT_EPOCHS} epochs over {len(dapt_train)} comments ...")
    total_steps = (len(ds["train"]) / DAPT_BATCH) * DAPT_EPOCHS
    args = TrainingArguments(
        output_dir="/content/dapt_ckpt",
        num_train_epochs=DAPT_EPOCHS,
        per_device_train_batch_size=DAPT_BATCH,
        learning_rate=DAPT_LR,
        warmup_steps=int(0.1 * total_steps),
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        save_strategy="no",
        logging_steps=200,
        seed=42,
        report_to="none",
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                      data_collator=DataCollatorForLanguageModeling(tok, mlm=True,
                                                                    mlm_probability=MLM_PROB))
    trainer.train()

    print("\nMeasuring perplexity AFTER adaptation ...")
    ppl_after = perplexity(model, tok, ds["eval"])
    print(f"  adapted MuRIL perplexity: {ppl_after:.2f}")

    DAPT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(DAPT_DIR)
    tok.save_pretrained(DAPT_DIR)

    results["dapt"] = {"variant": VARIANT, "base": BASE_MODEL,
                       "n_train": len(dapt_train), "n_eval": len(dapt_eval),
                       "epochs": DAPT_EPOCHS, "lr": DAPT_LR,
                       "ppl_before": round(ppl_before, 3), "ppl_after": round(ppl_after, 3),
                       "minutes": round((time.time() - t0) / 60, 2)}
    save_results(results)

    print(f"\nDAPT complete in {results['dapt']['minutes']:.1f} min.")
    print(f"  perplexity {ppl_before:.2f} -> {ppl_after:.2f} "
          f"({100 * (ppl_before - ppl_after) / ppl_before:+.1f}%)")
    print("  A large drop means the model adapted to the domain. That is necessary")
    print("  for DAPT to help downstream, but not sufficient - the task score decides.")

    del model, trainer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 4. Fine-tuning harness - identical to the sweep, so numbers are comparable
# ============================================================
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

def load_split(lang, split):
    df = pd.read_csv(DATA_DIR / f"{lang}_{split}_processed.csv")
    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.strip() != ""]
    assert set(df["label"].unique()) <= {0, 1}
    return df[["text", "label"]].reset_index(drop=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "hope_f1": f1_score(labels, preds, pos_label=1),
            "hope_precision": precision_score(labels, preds, pos_label=1, zero_division=0),
            "hope_recall": recall_score(labels, preds, pos_label=1, zero_division=0)}

_cache = {}
def get_data(model_path):
    key = str(model_path)
    if key in _cache:
        return _cache[key]
    tok = AutoTokenizer.from_pretrained(model_path)
    dfs = {s: load_split(LANG, s) for s in ("train", "dev", "test")}
    ds = {}
    for name, df in dfs.items():
        d = Dataset.from_pandas(df, preserve_index=False)
        ds[name] = d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                         batched=True, remove_columns=["text"])
    _cache[key] = (ds, dfs, tok)
    return _cache[key]

def train_one(model_path, lr, seed, stage):
    t0 = time.time()
    ds, dfs, tok = get_data(model_path)
    set_seed(seed)

    cw = compute_class_weight("balanced", classes=np.array([0, 1]),
                              y=dfs["train"]["label"].values)
    cw_t = torch.tensor(cw, dtype=torch.float32).cuda()

    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), num_labels=2)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=cw_t)
            return (loss, outputs) if return_outputs else loss

    ckpt = Path("/content/ckpt_dapt")
    shutil.rmtree(ckpt, ignore_errors=True)
    total_steps = (len(ds["train"]) / BATCH_SIZE) * EPOCHS

    args = TrainingArguments(
        output_dir=str(ckpt),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_steps=int(0.1 * total_steps),
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=500,
        seed=seed,
        report_to="none",
        disable_tqdm=True,
    )
    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["dev"],
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    curve = [round(h["eval_macro_f1"], 4) for h in trainer.state.log_history
             if "eval_macro_f1" in h]
    dev  = trainer.evaluate(ds["dev"])
    test = trainer.evaluate(ds["test"])

    rec = {"variant": VARIANT, "stage": stage, "lr": lr, "seed": seed,
           "dev_macro_f1": round(dev["eval_macro_f1"], 4),
           "test_macro_f1": round(test["eval_macro_f1"], 4),
           "test_hope_f1": round(test["eval_hope_f1"], 4),
           "curve": curve,
           "degenerate": max(curve) < 0.45,
           "best_epoch": int(np.argmax(curve)) + 1,
           "minutes": round((time.time() - t0) / 60, 1)}

    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return rec

print("Harness ready.")

In [ ]:
# ============================================================
# 5. Stage 1 - learning-rate grid on the DAPT model (selected on DEV)
# ============================================================
for lr in LR_GRID:
    if already_done(results, lr, GRID_SEED):
        print(f"SKIP  grid  lr={lr:g} seed={GRID_SEED}")
        continue
    print(f"RUN   grid  lr={lr:g} seed={GRID_SEED} ...")
    r = train_one(DAPT_DIR, lr, GRID_SEED, "grid")
    results["runs"].append(r); save_results(results)
    flag = "  <-- DEGENERATE" if r["degenerate"] else ""
    print(f"      dev {r['dev_macro_f1']:.4f} | test {r['test_macro_f1']:.4f} "
          f"| best epoch {r['best_epoch']}/{EPOCHS} | {r['minutes']:.1f}min{flag}")

grid = [r for r in results["runs"]
        if r["variant"] == VARIANT and r["stage"] == "grid" and not r["degenerate"]]
BEST_LR = max(grid, key=lambda r: r["dev_macro_f1"])["lr"] if grid else None
print(f"\nStage 1 complete. Best learning rate on dev: {BEST_LR}")
print("(The non-DAPT MuRIL chose 1e-5. If DAPT shifts this, that is itself worth reporting.)")

In [ ]:
# ============================================================
# 6. Stage 2 - best config at two further seeds
# ============================================================
assert BEST_LR is not None, "Stage 1 produced no usable run - check for degeneracy."

for seed in EXTRA_SEEDS:
    if already_done(results, BEST_LR, seed):
        print(f"SKIP  seed {seed}")
        continue
    print(f"RUN   seed {seed}  lr={BEST_LR:g} ...")
    r = train_one(DAPT_DIR, BEST_LR, seed, "seed")
    results["runs"].append(r); save_results(results)
    print(f"      dev {r['dev_macro_f1']:.4f} | test {r['test_macro_f1']:.4f} "
          f"| best epoch {r['best_epoch']}/{EPOCHS} | {r['minutes']:.1f}min")

print("\nStage 2 complete.")

In [ ]:
# ============================================================
# 7. Summary - did DAPT clear the noise floor?
# ============================================================
runs = [r for r in results["runs"] if r["variant"] == VARIANT
        and abs(r["lr"] - BEST_LR) < 1e-12]
vals = [r["test_macro_f1"] for r in runs]
mean = float(np.mean(vals))
std  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
gain = mean - BASELINE_MEAN

print("=" * 72)
print(f"DAPT - MuRIL - Tamil - variant '{VARIANT}'")
print("=" * 72)

if results.get("dapt"):
    d = results["dapt"]
    print(f"\nStage 0  adaptation on {d['n_train']} in-domain comments")
    print(f"         perplexity {d['ppl_before']:.2f} -> {d['ppl_after']:.2f}  "
          f"({100 * (d['ppl_before'] - d['ppl_after']) / d['ppl_before']:+.1f}%)")

print(f"\nStage 1  learning-rate grid (dev)")
for r in sorted([x for x in results["runs"] if x["variant"] == VARIANT and x["stage"] == "grid"],
                key=lambda x: -x["dev_macro_f1"]):
    mark = "  <- best dev" if abs(r["lr"] - BEST_LR) < 1e-12 else ""
    print(f"         lr={r['lr']:<7g} dev {r['dev_macro_f1']:.4f}  test {r['test_macro_f1']:.4f}{mark}")

print(f"\nStage 2  best config at {len(vals)} seeds")
print(f"         runs : {', '.join(f'{v:.4f}' for v in vals)}")
print(f"         mean : {mean:.4f} +/- {std:.4f}")

print(f"\n{'-' * 72}")
print(f"  DAPT MuRIL      {mean:.4f} +/- {std:.4f}")
print(f"  non-DAPT MuRIL  {BASELINE_MEAN:.4f} +/- {BASELINE_STD:.4f}")
print(f"  gain            {gain:+.4f}")
print(f"  noise floor     {SEED_SPREAD:.4f}   (Tamil seed spread from the sweep)")
print(f"{'-' * 72}")

if abs(gain) < SEED_SPREAD:
    print("\n  VERDICT: the gain is INSIDE the noise floor.")
    print("  This does not show DAPT helps, and it does not show it fails. It shows")
    print("  the effect, if any, is smaller than the run-to-run variation already")
    print("  measured. Report it as a null result with the bound stated, not as a")
    print("  failure - and note the corpus is small for DAPT.")
else:
    direction = "HELPS" if gain > 0 else "HURTS"
    print(f"\n  VERDICT: the gain EXCEEDS the noise floor - DAPT {direction} on Tamil.")
    print("  This is a measurable effect, reportable with the seed spread alongside it.")

print(f"\nSaved -> metrics/dapt_results.json  ({len(results['runs'])} runs)")